# **Recurrent Neural Network**

## Modules

In [15]:
import os
import re
import numpy as np
import urllib.request
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [3]:
# Checking if our tensorflow is available or not
print("Tensorflow Version: ", tf.__version__)
print("Available GPU's: ", tf.config.list_physical_devices("GPU"))
print("Available CPU's: ", tf.config.list_physical_devices("CPU"))

Tensorflow Version:  2.21.0
Available GPU's:  []
Available CPU's:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


### Dataset

In [4]:
url = (
    "https://raw.githubusercontent.com/karpathy/"
    "char-rnn/master/data/tinyshakespeare/input.txt"
)
data_path = "data.txt"
if not os.path.exists(data_path):
    urllib.request.urlretrieve(url, data_path)
    print("Dataset downloaded successfully!")
with open(data_path, "r", encoding="utf-8") as file:
    text = file.read()
print("Dataset loaded successfully!")

Dataset loaded successfully!


### Dataset Information

In [ ]:
print("Total characters:", len(text))
print("Total lines:", len(text.splitlines()))
print("File size:", round(os.path.getsize(data_path) / 1024, 2), "KB")
print("\nFirst 1000 characters:\n")
print(text[:100])

Total characters: 1115394
Total lines: 40000
File size: 1089.25 KB

First 1000 characters:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


### **Text preprocessing**

In [ ]:
# Convert all text to lowercase
clean_text = text.lower()

# Replace multiple whitespace characters with a single space
clean_text = re.sub(r"\s+", " ", clean_text).strip()

print("Original text:")
print(text[:100])

print("\nCleaned text:")
print(clean_text[:100])

Original text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

Cleaned text:
first citizen: before we proceed any further, hear me speak. all: speak, speak. first citizen: you a


### **Tokenization**

In [ ]:
# Extract words and punctuation as separate tokens
tokens = re.findall(r"\b\w+\b|[^\w\s]", clean_text)

print("First 30 tokens:")
print(tokens[:30])

print("\nTotal tokens:", len(tokens))
print("Unique tokens:", len(set(tokens)))

First 30 tokens:
['first', 'citizen', ':', 'before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', 'all', ':', 'speak', ',', 'speak', '.', 'first', 'citizen', ':', 'you', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than']

Total tokens: 262927
Unique tokens: 11466


### **Vocabulary**

In [11]:
# Create a vocabulary of unique tokens
vocabulary = sorted(set(tokens))

# Assign a unique integer ID to each token
word_to_id = {
    word: index
    for index, word in enumerate(vocabulary)
}

# Create the reverse mapping
id_to_word = {
    index: word
    for word, index in word_to_id.items()
}

# Convert every token into its numerical ID
token_ids = np.array(
    [word_to_id[word] for word in tokens],
    dtype=np.int32
)

vocab_size = len(vocabulary)

print("Vocabulary size:", vocab_size)
print("First 20 vocabulary words:", vocabulary[:20])
print("First 20 token IDs:", token_ids[:20])

Vocabulary size: 11466
First 20 vocabulary words: ['!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'a', 'abandon', 'abase', 'abate', 'abated', 'abbey', 'abbot', 'abed', 'abel']
First 20 token IDs: [ 3820  1759     8   841 11061  7619   413  4177     4  4716  6150  9240
     6   291     8  9240     4  9240     6  3820]


### **Sequence Length**

In [12]:
SEQUENCE_LENGTH = 10
print("Input sequence length:", SEQUENCE_LENGTH)

Input sequence length: 10


In [ ]:
# Training Examples

# Create input sequences and next-token targets
X = []
y = []

for i in range(SEQUENCE_LENGTH, len(token_ids)):
    X.append(token_ids[i - SEQUENCE_LENGTH:i])
    y.append(token_ids[i])

# Convert lists to NumPy arrays
X = np.array(X, dtype=np.int32)
y = np.array(y, dtype=np.int32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFirst input IDs:", X[0])
print("Target ID:", y[0])
print("\nInput words:")
print([id_to_word[int(idx)] for idx in X[0]])
print("Target word:", id_to_word[int(y[0])])

X shape: (262917, 10)
y shape: (262917,)

First input IDs: [ 3820  1759     8   841 11061  7619   413  4177     4  4716]
Target ID: 6150

Input words:
['first', 'citizen', ':', 'before', 'we', 'proceed', 'any', 'further', ',', 'hear']
Target word: me


### **Train-Validation Split**

In [14]:
# Use the first 90% of sequences for training
split_index = int(len(X)*0.90)
X_train = X[:split_index]
y_train = y[:split_index]

X_val = X[split_index:]
y_val = y[split_index:]

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

Training samples: 236625
Validation samples: 26292
X_train shape: (236625, 10)
y_train shape: (236625,)
X_val shape: (26292, 10)
y_val shape: (26292,)


In [16]:
model = Sequential([
    Embedding(
        input_dim= vocab_size,
        output_dim= 64
    ),
    SimpleRNN(
        units = 128
    ),
    Dense(
        units = vocab_size,
        activation= "softmax"
    )
])
model.build(input_shape=(None, SEQUENCE_LENGTH))
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 10, 64)         │       733,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 11466)          │     1,479,114 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,237,642 (8.54 MB)

 Trainable params: 2,237,642 (8.54 MB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# Compile

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"]
)
print("Model compiled successfully!")

Model compiled successfully!


In [18]:
# Training

with tf.device("/CPU:0"):
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=5,
        batch_size=128,
        shuffle=False
    )

Epoch 1/5
1849/1849 ━━━━━━━━━━━━━━━━━━━━ 22s 12ms/step - loss: 5.8548 - sparse_categorical_accuracy: 0.1157 - val_loss: 5.5468 - val_sparse_categorical_accuracy: 0.1442
Epoch 2/5
1849/1849 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 5.1154 - sparse_categorical_accuracy: 0.1613 - val_loss: 5.4390 - val_sparse_categorical_accuracy: 0.1569
Epoch 3/5
1849/1849 ━━━━━━━━━━━━━━━━━━━━ 26s 14ms/step - loss: 4.8134 - sparse_categorical_accuracy: 0.1792 - val_loss: 5.4625 - val_sparse_categorical_accuracy: 0.1571
Epoch 4/5
1849/1849 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 4.6044 - sparse_categorical_accuracy: 0.1909 - val_loss: 5.5123 - val_sparse_categorical_accuracy: 0.1573
Epoch 5/5
1849/1849 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 4.4349 - sparse_categorical_accuracy: 0.1996 - val_loss: 5.5768 - val_sparse_categorical_accuracy: 0.1555


In [19]:
# Traning results

print("Final training loss:", history.history["loss"][-1])
print("Final validation loss:", history.history["val_loss"][-1])

print(
    "Final training accuracy:",
    history.history["sparse_categorical_accuracy"][-1]
)

print(
    "Final validation accuracy:",
    history.history["val_sparse_categorical_accuracy"][-1]
)

Final training loss: 4.434885501861572
Final validation loss: 5.576782703399658
Final training accuracy: 0.19962388277053833
Final validation accuracy: 0.15548455715179443


In [20]:
# Predicting the next 3 words

def predict_next_words(seed_text, num_words=3):
    # Convert input text into tokens
    input_tokens = re.findall(
        r"\b\w+\b|[^\w\s]",
        seed_text.lower()
    )

    # Check for words that are not in our vocabulary
    unknown_tokens = [
        token for token in input_tokens
        if token not in word_to_id
    ]

    if unknown_tokens:
        raise ValueError(
            f"Unknown tokens: {unknown_tokens}. "
            "Please use words present in the training vocabulary."
        )

    # We need at least SEQUENCE_LENGTH tokens
    if len(input_tokens) < SEQUENCE_LENGTH:
        raise ValueError(
            f"Please enter at least {SEQUENCE_LENGTH} tokens."
        )

    # Convert words to token IDs
    input_ids = [
        word_to_id[token] for token in input_tokens
    ]

    generated_words = []

    for _ in range(num_words):
        # Keep the most recent 10 tokens
        context = input_ids[-SEQUENCE_LENGTH:]

        # Model expects shape: (batch_size, sequence_length)
        context = np.array([context], dtype=np.int32)

        # Get predicted probabilities
        probabilities = model.predict(
            context,
            verbose=0
        )[0]

        # Select the token with the highest probability
        predicted_id = int(np.argmax(probabilities))

        # Convert token ID back into a word or punctuation
        predicted_token = id_to_word[predicted_id]

        generated_words.append(predicted_token)

        # Feed prediction into the next iteration
        input_ids.append(predicted_id)

    return generated_words

In [21]:
seed_text = (
    "the king was standing in front of the great castle"
)

predictions = predict_next_words(seed_text, num_words=3)

print("Input:", seed_text)
print("Predicted next 3 tokens:", predictions)
print("Generated text:", seed_text + " " + " ".join(predictions))

Input: the king was standing in front of the great castle
Predicted next 3 tokens: ['.', 'gremio', ':']
Generated text: the king was standing in front of the great castle . gremio :


- **NOTE** - here our project ended but as we can see the output will try to predict the good words.

In [22]:
seed_text = "the king was standing in front of the great castle"

input_tokens = re.findall(
    r"\b\w+\b|[^\w\s]",
    seed_text.lower()
)

input_ids = [word_to_id[token] for token in input_tokens]
context = np.array(
    [input_ids[-SEQUENCE_LENGTH:]],
    dtype=np.int32
)

probabilities = model.predict(context, verbose=0)[0]

top_indices = np.argsort(probabilities)[-5:][::-1]

for idx in top_indices:
    print(
        f"{id_to_word[int(idx)]:<15} "
        f"Probability: {probabilities[idx]:.4f}"
    )

.               Probability: 0.2597
,               Probability: 0.1287
:               Probability: 0.0850
and             Probability: 0.0624
;               Probability: 0.0569


now we can get that why our model was predicting the full stop as the next word as the probability of it is 0.25.

In [23]:
seed_text = "the king was standing in front of the great castle"

for i in range(5):
    predictions = predict_next_words(seed_text, num_words=3)
    print(f"Attempt {i + 1}: {' '.join(predictions)}")

Attempt 1: . gremio :
Attempt 2: . gremio :
Attempt 3: . gremio :
Attempt 4: . gremio :
Attempt 5: . gremio :


In [ ]:
def sample_next_words(seed_text, num_words=3, temperature=0.8, top_k=10):
    input_tokens = re.findall(
        r"\b\w+\b|[^\w\s]",
        seed_text.lower()
    )

    unknown_tokens = [
        token for token in input_tokens
        if token not in word_to_id
    ]

    if unknown_tokens:
        raise ValueError(f"Unknown tokens: {unknown_tokens}")

    if len(input_tokens) < SEQUENCE_LENGTH:
        raise ValueError(
            f"Please enter at least {SEQUENCE_LENGTH} tokens."
        )

    input_ids = [word_to_id[token] for token in input_tokens]
    generated_tokens = []

    for _ in range(num_words):
        context = np.array(
            [input_ids[-SEQUENCE_LENGTH:]],
            dtype=np.int32
        )

        probabilities = model.predict(context, verbose=0)[0]

        # Adjust probabilities using temperature
        probabilities = np.log(
            np.clip(probabilities, 1e-10, 1.0)
        ) / temperature

        probabilities = np.exp(
            probabilities - np.max(probabilities)
        )
        probabilities = probabilities / probabilities.sum()

        # Keep only the top-k candidates
        top_indices = np.argsort(probabilities)[-top_k:]
        top_probabilities = probabilities[top_indices]
        top_probabilities = top_probabilities / top_probabilities.sum()

        # Randomly select one candidate based on probability
        predicted_id = int(
            np.random.choice(top_indices, p=top_probabilities)
        )

        generated_tokens.append(id_to_word[predicted_id])
        input_ids.append(predicted_id)

    return generated_tokens

In [ ]:
# Testing the new function

seed_text = "the king was standing in front of the great castle"

for i in range(5):
    predictions = sample_next_words(
        seed_text,
        num_words=3,
        temperature=0.8,
        top_k=10
    )
    print(f"Attempt {i + 1}: {' '.join(predictions)}")

Attempt 1: and a thing
Attempt 2: . o your
Attempt 3: . katharina ,
Attempt 4: , and i
Attempt 5: and in all


## Final Output and Experiment Results

The Vanilla RNN successfully generated three consecutive tokens using the Tiny Shakespeare dataset, achieving 19.96% training accuracy and 15.55% validation accuracy.

- **Top-5 Probability Analysis:** Identified the most likely next tokens and their probabilities.
- **Greedy Decoding:** Selected the highest-probability token at each step, producing consistent but sometimes incoherent text.
- **Top-k Sampling:** Generated varied outputs by sampling from the top 10 candidate tokens, but the results were not always meaningful.

**Conclusion:** The project demonstrated next-token prediction and autoregressive text generation, along with how different decoding strategies affect the generated output.